In [2]:
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

In [42]:
from pathlib import Path

# Resolves a relative path to an absolute path
path = Path("example.txt").resolve()
print(path) 


C:\Users\elbou\Desktop\contractAssistant\contractAssistant\ingestion\example.txt


In [6]:
from typing import List, Any
from langchain_core.documents import Document
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import AcceleratorDevice, AcceleratorOptions
import logging
logger = logging.getLogger(__name__)


def load_all_document(data_dir:str)->List[Document]:
    """load all documents from data directory"""

    data_path = Path(data_dir).resolve()
    logger.info(f"data path: {data_path}")

    # configure docling pipeline
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False
    pipeline_options.do_table_structure = True
    pipeline_options.accelerator_options = AcceleratorOptions(
        device=AcceleratorDevice.CUDA
    )
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options
            )
        }
    )
    pdf_files = list(data_path.glob("**/*.pdf"))
    logger.info(f"Found {len(pdf_files)} pdf files")
    documents = []

    for pdf_file in pdf_files:
        try:
            logger.info(f"Loading pdf: {pdf_file}")
            # convert with docling
            result = converter.convert(str(pdf_file))
            doc = result.document

            ## split by pages
            pages = sorted(doc.pages.keys()) if doc.pages else []

            if pages:
                for page in pages:
                    page_text = doc.export_to_markdown(page_no=page)
                    if not page_text.strip():
                        continue

                    documents.append(Document(
                        page_content=page_text,
                        metadata={
                            "file_type":     "pdf",
                            "source_file":   pdf_file.name,
                            "contract_type": pdf_file.parent.name,  # ← folder = contract type
                            "page":          page,
                            "source":        str(pdf_file),
                        }
                    ))
            else:
                # Fallback — export full document as one chunk if no pages
                full_text = doc.export_to_markdown()
                if full_text.strip():
                    documents.append(Document(
                        page_content=full_text,
                        metadata={
                            "file_type":     "pdf",
                            "source_file":   pdf_file.name,
                            "contract_type": pdf_file.parent.name,
                            "page":          0,
                            "source":        str(pdf_file),
                        }
                    ))

            logger.debug(f"Loaded {pdf_file.name}")
        except Exception as e:
            logger.error(f"Failed to load {pdf_file}: {e}")
    
    logger.info(f"Loaded {len(documents)} documents")  
    return documents


In [7]:
docs = load_all_document("../data/testing_2")

2026-04-01 18:33:36,641 | INFO | data path: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\testing_2
2026-04-01 18:33:36,694 | INFO | Found 1 pdf files
2026-04-01 18:33:36,696 | INFO | Loading pdf: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\testing_2\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf
2026-04-01 18:33:36,698 | INFO | detected formats: [<InputFormat.PDF: 'pdf'>]
2026-04-01 18:33:36,702 | INFO | Going to convert document batch...
2026-04-01 18:33:36,704 | INFO | Initializing pipeline for StandardPdfPipeline with options hash 6e159a8263d407a101b59338024580e4
2026-04-01 18:33:36,706 | INFO | Accelerator device: 'cuda:0'
2026-04-01 18:33:36,710 | DEBUG | close.started
2026-04-01 18:33:36,710 | DEBUG | close.complete
2026-04-01 18:33:36,710 | DEBUG | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=None socket_options=None
2026-04-01 18:33:38,003 | DEBUG | connect_tcp.complete r

In [8]:
docs

[Document(metadata={'file_type': 'pdf', 'source_file': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'contract_type': 'testing_2', 'page': 1, 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\testing_2\\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'}, page_content='## Exhibit 10.27\n\n## MARKETING AFFILIATE AGREEMENT\n\nBetween:\n\n## Birch First Global Investments Inc.\n\nAnd\n\n## Mount Knowledge Holdings Inc.\n\nDated: May 8, 2014\n\n1'),
 Document(metadata={'file_type': 'pdf', 'source_file': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'contract_type': 'testing_2', 'page': 2, 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\testing_2\\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'}, page_content="This\xa0Marketing\xa0Affiliate\xa0Agreement\xa0(the\xa0' Agreement ')\

In [10]:
print(f"Type of doc: {type(docs[0])}, Value: {docs[0]}")

Type of doc: <class 'langchain_core.documents.base.Document'>, Value: page_content='Exhibit 10.6
TRADEMARK LICENSE AGREEMENT
This TRADEMARK LICENSE AGREEMENT (this “Agreement”) is made and effective as of [ ] day of [ ], 2020 (“Effective Date”), by and
between Palmer Square Capital Management LLC, a Delaware limited liability company (the “Licensor”), and Palmer Square Capital BDC Inc., a
corporation organized under the laws of the State of Maryland (the “Licensee”) (each a “party,” and collectively, the “parties”).
RECITALS
WHEREAS, Licensee is a newly organized, externally managed, closed-end, non-diversified management investment company that intends
to elect to be regulated as a business development company under the Investment Company Act of 1940, as amended (the “1940 Act”);
WHEREAS, Licensor, together with its affiliates, provides investment management, investment consultation and investment advisory
services;
WHEREAS, Licensor and its affiliates, including Palmer Square BDC Adv

### enrich metadata

In [9]:
import pandas as pd

## clean date fynction

def clean_date_human_display(raw_data:str)->str:
    # 1. Clean the string: Remove brackets and whitespace
    raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
    # 2. Convert to a datetime object
    date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
    # 3. Format it back to your desired string style (MM/DD/YY)
    if pd.notnull(date_obj):
    # %m/%d/%y gives you 07/19/12. 
    # If you want 7/19/12 (no leading zero), use .strftime('%#m/%#d/%y') on Windows
        clean_date_str = date_obj.strftime('%m/%d/%y')
    else:
        clean_date_str = "Unknown" # Or keep it as None/nan
    return clean_date_str

## clean this for qdrant database
def clean_date_iso(raw_data:str):
    # 1. Clean the string: Remove brackets and whitespace
    raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
    # 2. Convert to a datetime object
    date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
    # 3. Format it back to ISO format (YYYY-MM-DD)
    if pd.notnull(date_obj):
        clean_date_str = date_obj.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        clean_date_str = "Unknown" 
    return clean_date_str

def clear_nan(raw_value):
    if pd.isna(raw_value):
        return "Unknown"
    else:
        return raw_value
def clear_brackets_and_nan(raw_value):
    ## ckear values with brackets and also the nan values
    if pd.isna(raw_value):
        return "Unknown"
    else:
        new_value =  str(raw_value).replace("[", "").replace("]", "").strip()
        if new_value == "":
            return "Unknown"
        return new_value
        


In [10]:
import re
import logging
import pandas as pd

logger = logging.getLogger(__name__)


def enrich_metadata(documents):

    logger.info(f"strat enriching metadata for {len(documents)} documents")
    data = pd.read_csv("../data/master_clauses.csv")

    for doc in documents:
        for index, row in data.iterrows():
            if doc.metadata.get("source_file") == row["Filename"]:
                #parties:
                parties = re.findall(r'([^;()]+)(?:\s*\(|$)', row["Parties-Answer"])
                parties = [party.strip() for party in parties if party.strip()] 
                party_1 = parties[0] if len(parties) > 0 else "Unknown"
                party_2 = parties[1] if len(parties) > 1 else "Unknown"
                doc.metadata.update({
                    "party_1":party_1,
                    "party_2":party_2,
                    "contract_type":row["Document Name-Answer"],
                    "agreement_date":clean_date_iso(row["Agreement Date-Answer"]),
                    "effective_date":clean_date_iso(row["Effective Date-Answer"]),
                    "expiration_date":clean_date_iso(row["Expiration Date-Answer"]),
                    "agreement_date_human_display":clean_date_human_display(row["Agreement Date-Answer"]),
                    "effective_date_human_display":clean_date_human_display(row["Effective Date-Answer"]),
                    "expiration_date_human_display":clean_date_human_display(row["Expiration Date-Answer"]),
                    "notice_period_to_terminate": clear_brackets_and_nan(row["Notice Period To Terminate Renewal- Answer"]),
                    "renewl_term":clear_nan(row["Renewal Term-Answer"]),
                    "governing_law":row["Governing Law-Answer"]

                })
    logger.info(f"metadata enriched successfully")
    return documents


In [11]:
docs = enrich_metadata(docs)

2026-04-01 18:35:20,578 | INFO | strat enriching metadata for 16 documents
2026-04-01 18:35:22,209 | INFO | metadata enriched successfully


In [12]:
docs

[Document(metadata={'file_type': 'pdf', 'source_file': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'contract_type': 'MARKETING AFFILIATE AGREEMENT', 'page': 1, 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\testing_2\\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'party_1': 'Birch First Global Investments Inc.', 'party_2': 'Mount Kowledge Holdings Inc.', 'agreement_date': '2014-05-08T00:00:00Z', 'effective_date': 'Unknown', 'expiration_date': '2014-12-31T00:00:00Z', 'agreement_date_human_display': '05/08/14', 'effective_date_human_display': 'Unknown', 'expiration_date_human_display': '12/31/14', 'notice_period_to_terminate': '30 days', 'renewl_term': 'successive 1 year', 'governing_law': 'Nevada'}, page_content='## Exhibit 10.27\n\n## MARKETING AFFILIATE AGREEMENT\n\nBetween:\n\n## Birch First Global Investments Inc.\n\nAnd\n\n## Mount Knowledge Holdings Inc.\n\nDated: Ma

### chunking

In [16]:
import logging
import re
from typing import List
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

logger = logging.getLogger(__name__)

SECTION_PATTERN = re.compile(r"\n\s*(\d+)\.\s+[A-Z][^\n]+")


def split_by_sections(text: str):
    """
    Split contract text by legal section headers like:
    1. Services
    2. Compensation
    """

    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        return [text]
    sections = []
    for i, match in enumerate(matches):
        header_info = f"Section {match.group(0)}:\n"

        start = match.start()
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)
        section_content = text[start:end].strip()
        sections.append(f"{header_info}{section_content}")
    return sections


def chunk_contract_documents(documents: List[Document]) -> List[Document]:
    """
    Chunk contracts using section-aware chunking,
    then semantic chunking for large sections.
    """
    ## add model_kwargs and encode_kwargs to force using GPU
    model_kwargs = {"device":"cuda"}
    encode_kwargs = {"normalize_embeddings": True}
    embeddings = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-large-en-v1.5",
        model_kwargs = model_kwargs,
        encode_kwargs = encode_kwargs
        
    )

    semantic_chunker = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90,
    )
    
    logger.info(f"initialise embedding model for Semantic Chunker ")
    all_chunks = []
    logger.info("start chunking ...")
    for doc in documents:
        text = doc.page_content
        # split by contract sections
        sections = split_by_sections(text)
        for section in sections:
            if len(section) < 2000:
                # small section → keep as one chunk
                chunk = Document(
                    page_content= section,
                    metadata=doc.metadata.copy()
                )
                all_chunks.append(chunk)
            else:
                # large section → semantic chunking
                semantic_chunks = semantic_chunker.create_documents(
                    [section],
                    [doc.metadata]
                )
                all_chunks.extend(semantic_chunks)


    all_chunks =  [doc for doc in all_chunks if len(doc.page_content) > 25]
    logger.info(f"Created {len(all_chunks)} chunks")
    return all_chunks


In [17]:
chunks = chunk_contract_documents(docs)

2026-04-01 18:39:53,675 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-04-01 18:39:53,685 | DEBUG | close.started
2026-04-01 18:39:53,692 | DEBUG | close.complete
2026-04-01 18:39:53,696 | DEBUG | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-04-01 18:39:53,769 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000019006EB5010>
2026-04-01 18:39:53,773 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000018DFD4C34A0> server_hostname='huggingface.co' timeout=10
2026-04-01 18:39:53,814 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000019006EB6E10>
2026-04-01 18:39:53,821 | DEBUG | send_request_headers.started request=<Request [b'HEAD']>
2026-04-01 18:39:53,827 | DEBUG | send_request_headers.complete
2026-04-01 18:39:53,832 | DEBUG | send_request_body.started request=<Request [b'HEAD']>
2026-04-01 1

2026-04-01 18:39:54,020 | DEBUG | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'349'), (b'Connection', b'keep-alive'), (b'Date', b'Thu, 07 Aug 2025 14:51:14 GMT'), (b'ETag', b'"952a9b81c0bfd99800fabf352f69c7ccd46c5e43"'), (b'X-Powered-By', b'huggingface-moon'), (b'cross-origin-opener-policy', b'same-origin'), (b'Referrer-Policy', b'strict-origin-when-cross-origin'), (b'X-Request-Id', b'Root=1-6894bd62-6818d5c26bb086fa4a8fe615;10f07242-5e52-410d-802b-09b39db9c9d5'), (b'Access-Control-Allow-Origin', b'https://huggingface.co'), (b'Access-Control-Expose-Headers', b'X-Repo-Commit,X-Request-Id,X-Error-Code,X-Error-Message,X-Total-Count,ETag,Link,Accept-Ranges,Content-Range,X-Linked-Size,X-Linked-ETag,X-Xet-Hash'), (b'Set-Cookie', b'token=; Path=/; Expires=Thu, 01 Jan 1970 00:00:00 GMT; Secure; SameSite=None'), (b'Set-Cookie', b'token=; Domain=huggingface.co; Path=/; Expires=Thu, 01 Jan 1970 00:

In [18]:
chunks

[Document(metadata={'file_type': 'pdf', 'source_file': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'contract_type': 'MARKETING AFFILIATE AGREEMENT', 'page': 1, 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\testing_2\\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'party_1': 'Birch First Global Investments Inc.', 'party_2': 'Mount Kowledge Holdings Inc.', 'agreement_date': '2014-05-08T00:00:00Z', 'effective_date': 'Unknown', 'expiration_date': '2014-12-31T00:00:00Z', 'agreement_date_human_display': '05/08/14', 'effective_date_human_display': 'Unknown', 'expiration_date_human_display': '12/31/14', 'notice_period_to_terminate': '30 days', 'renewl_term': 'successive 1 year', 'governing_law': 'Nevada'}, page_content='## Exhibit 10.27\n\n## MARKETING AFFILIATE AGREEMENT\n\nBetween:\n\n## Birch First Global Investments Inc.\n\nAnd\n\n## Mount Knowledge Holdings Inc.\n\nDated: Ma

In [1]:
# delete collection

from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)
client.delete_collection(collection_name="contracts")  # deletes old collection

c:\Users\elbou\Desktop\contractAssistant\contractAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### store in the qdrant db

In [26]:
## creating data
import logging
import hashlib
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models

logger = logging.getLogger(__name__)


## embeed then store
class QdrantStore:

    def __init__(
        self,
        dense_model_name: str = "BAAI/bge-large-en-v1.5",
        sparse_model_name: str = "Qdrant/bm25",
    ):
        self.dense_model_name = dense_model_name
        self.sparse_model_name = sparse_model_name
        ## add model_kwargs={'device': 'cuda'} for GPU Enabling
        self.dense_model = SentenceTransformer(self.dense_model_name,device="cuda")
        ## add providers=["CUDAExecutionProvider"] for GPU Enabling
        self.sparse_model = SparseTextEmbedding(model_name=self.sparse_model_name, providers=["CUDAExecutionProvider"])
        self.client = QdrantClient(host="localhost", port=6333, timeout=60)
        self.collection_name = "contracts_v1"
        self.creat_collection()

    def creat_collection(self):
        if not self.client.collection_exists(collection_name=self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    "dense": models.VectorParams(
                        size=1024, distance=models.Distance.COSINE
                    )
                },
                sparse_vectors_config={
                    "sparse": models.SparseVectorParams(
                        index=models.SparseIndexParams(on_disk=False)
                    )
                },
                timeout=60
            )
    ## generate IDs
    def generate_doc_id(self, source: str, content: str):
        unique_string = f"{source}:{content}"
        return hashlib.sha256(unique_string.encode()).hexdigest()[:32]

    def embedde_chunks_and_store(self, chunks):

        logger.info(f"generating embedding {len(chunks)} chunks ...")

        texts = [chunk.page_content for chunk in chunks]

        dense_embeddings = self.dense_model.encode(
            texts, normalize_embeddings=True, batch_size=16, show_progress_bar=True
        )
        sparse_embeddings = list(self.sparse_model.embed(texts))

        # Focuses on Semantic Direction: By removing the effect of vector magnitude, normalization ensures that similarity is based on semantic meaning rather than frequency or length.
        logger.info(f"embeddings shape: {dense_embeddings.shape}")

        points = []
        batch_size = 64

        for chunk, dense_embedding, sparse_embedding in zip(
            chunks, dense_embeddings, sparse_embeddings
        ):
            ## generating ID
            source = chunk.metadata.get("source_file","uknown")
            doc_id = self.generate_doc_id(source,chunk.page_content)

            ## getting general context
            contract_type =chunk.metadata.get("contract_type","Legal Document")
            parties = f"{chunk.metadata.get('party_1','')} and {chunk.metadata.get('party_2','')}"
            
            #context header
            contract_info = f"Type: {chunk.metadata.get('contract_type', 'Legal')}"
            points.append(
                models.PointStruct(
                    # id=uuid.uuid4(),
                    id=doc_id,
                    vector={
                        "dense": dense_embedding.tolist(),
                        "sparse": models.SparseVector(
                            indices=sparse_embedding.indices,
                            values=sparse_embedding.values,
                        ),
                    },
                    payload={
                        # "text": f"Document: {contract_type} between {parties}, \n Context: {chunk.page_content}",

                        "text": chunk.page_content,
                        "context_header": f"{source} from {contract_info}",
                        **chunk.metadata,

                        # "source": chunk.metadata.get("source_file", ""),
                        # "file_type": chunk.metadata.get("file_type", ""),
                        # "page": chunk.metadata.get("page", ""),
                        # "contract_type": chunk.metadata.get("contract_type", ""),
                        # # "clause_type": chunk.metadata.get("Clause_type", ""),
                        # "agreement_date":chunk.metadata.get("agreement_date", ""),
                        # "effective_date": chunk.metadata.get("effective_date", ""),
                        # "expiration_date": chunk.metadata.get("expiration_date", ""),
                        # "agreement_date_human_display":chunk.metadata.get("agreement_date_human_display", ""),
                        # "effective_date_human_display":chunk.metadata.get("effective_date_human_display", ""),
                        # "expiration_date_human_display":chunk.metadata.get("expiration_date_human_display", ""),
                        # "party_1": chunk.metadata.get("party_1", ""),
                        # "party_2": chunk.metadata.get("party_2", ""),
                        # "notice_period_to_terminate": chunk.metadata.get("notice_period_to_terminate", ""),
                        # "renewl_term": chunk.metadata.get("renewl_term", ""),
                        # "governing_law": chunk.metadata.get("governing_law", ""),
                    },
                )
            )
            if len(points) > batch_size:
                self.client.upsert(collection_name=self.collection_name, points=points)
                logger.info(f"upsertted {len(points)}..")
                points = []
        if points:
            self.client.upsert(collection_name=self.collection_name, points=points)
            logger.info(f"upsertted final {len(points)} points.")

        logger.info("chunks stored in Qdrant.")


In [27]:
store = QdrantStore()
store.embedde_chunks_and_store(chunks)

2026-04-01 19:18:02,606 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-04-01 19:18:02,619 | DEBUG | close.started
2026-04-01 19:18:02,628 | DEBUG | close.complete
2026-04-01 19:18:02,635 | DEBUG | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None


2026-04-01 19:18:03,912 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000018DBDCAC690>
2026-04-01 19:18:03,916 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000018DFD4C34A0> server_hostname='huggingface.co' timeout=10
2026-04-01 19:18:04,096 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000018DBDCAC150>
2026-04-01 19:18:04,103 | DEBUG | send_request_headers.started request=<Request [b'HEAD']>
2026-04-01 19:18:04,114 | DEBUG | send_request_headers.complete
2026-04-01 19:18:04,123 | DEBUG | send_request_body.started request=<Request [b'HEAD']>
2026-04-01 19:18:04,127 | DEBUG | send_request_body.complete
2026-04-01 19:18:04,131 | DEBUG | receive_response_headers.started request=<Request [b'HEAD']>
2026-04-01 19:18:04,297 | DEBUG | receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Conte

### rewrite query

In [1]:

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
import os 

load_dotenv()

class QueryRewriting:
    def __init__(self, model_name:str="llama-3.3-70b-versatile"): #
        self.model_name = model_name

        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("GROQ_API_KEY not found in environment variables")
        self.llm = ChatGroq(groq_api_key = groq_api_key, model_name = self.model_name, temperature=0)

        self.prompt = PromptTemplate(
            input_variables=["original_query"],
            template="""You are an expert legal assistant helping retrieve information \
                    from enterprise contracts.

                    Reformulate the user query below to be more precise and retrieval-friendly.
                    Return ONLY the rewritten query, nothing else.

                    Original query: {original_query}
                    Rewritten query:"""
        )

        self.chain = self.prompt | self.llm
        print(f"[INFO] Groq LLM initialized {self.model_name}")

    def rewrite_query(self,original_query:str):
        if not original_query.strip():
            raise ValueError("original query cannot be empty")
        response = self.chain.invoke({"original_query": original_query})
        return response.content.strip()


c:\Users\elbou\Desktop\contractAssistant\contractAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
original_query = "is this agreement done between Birch First Global Investments Inc. and Mount Kowledge Holdings Inc. "
rewriter = QueryRewriting()
rewriten = rewriter.rewrite_query(original_query)

[INFO] Groq LLM initialized llama-3.3-70b-versatile


In [3]:
rewriten
#'Identify the parties executing the agreement in the contract document.'

"What are the parties involved in this agreement, specifically is it between Birch First Global Investments Inc. and Mount Kowledge Holdings Inc., as listed in the contract's preamble or signature block."

### filters

In [ ]:

# from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
# from langchain_classic.chains.query_constructor.schema import AttributeInfo
# from langchain_qdrant import  QdrantVectorStore
# import os
# import logging
# from dotenv import load_dotenv

# load_dotenv()
# logger = logging.getLogger(__name__)
# embeddings = HuggingFaceEmbeddings(
#         model_name = "BAAI/bge-large-en-v1.5",
#         model_kwargs = {"device":"cpu"},
#         encode_kwargs = {"normalize_embeddings": True}
#     )




2026-04-01 20:44:15,126 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-04-01 20:44:15,127 | DEBUG | close.started
2026-04-01 20:44:15,127 | DEBUG | close.complete
2026-04-01 20:44:15,128 | DEBUG | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-04-01 20:44:17,266 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000018E421EE1D0>
2026-04-01 20:44:17,268 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000018DFD4C34A0> server_hostname='huggingface.co' timeout=10
2026-04-01 20:44:21,573 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000018E3487E810>
2026-04-01 20:44:21,576 | DEBUG | send_request_headers.started request=<Request [b'HEAD']>
2026-04-01 20:44:21,578 | DEBUG | send_request_headers.complete
2026-04-01 20:44:21,579 | DEBUG | send_request_body.started request=<Request [b'HEAD']>
2026-04-01 2

### Query constructor

In [ ]:
from langchain_classic.chains.query_constructor.base import load_query_constructor_runnable, load_query_constructor_chain
from langchain_community.query_constructors.qdrant import QdrantTranslator
from langchain_classic.chains.query_constructor.ir import Comparison, Operation
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from qdrant_client import models
import os
from langchain_groq import ChatGroq
def flatten_metadata_values(node):
    """
    Recursively finds dictionaries like {'date': '...'} in the AST 
    and replaces them with the actual string value.
    """
    if isinstance(node, Comparison):
        # If the value is that problematic dictionary, grab just the date string
        if isinstance(node.value, dict) and 'date' in node.value:
            node.value = node.value['date']
            
    elif isinstance(node, Operation):
        # Recursively check all arguments in AND/OR operations
        for arg in node.arguments:
            flatten_metadata_values(arg)
    return node
def remove_content_prefix(node):
    """
    Recursively removes 'content.' from any attribute names in the AST.
    """
    if hasattr(node, 'attribute') and node.attribute.startswith("content."):
        node.attribute = node.attribute.replace("content.", "")
            
    if hasattr(node, 'arguments'):
        for arg in node.arguments:
            remove_content_prefix(arg)
    return node

def finalize_qdrant_keys(filter_obj):
    """
    Recursively removes 'content.' and any leading dots from 
    FieldCondition keys to match a flat Qdrant payload.
    """
    if filter_obj is None:
        return None

    # Helper to clean a list of conditions (must, should, must_not)
    def clean_list(conditions):
        if not conditions:
            return
        for condition in conditions:
            # 1. If it's a direct FieldCondition, fix the key
            if isinstance(condition, models.FieldCondition):
                # Remove 'content.' if present
                new_key = condition.key.replace("content.", "")
                # Remove a leading dot if it exists (e.g., '.party_1' -> 'party_1')
                if new_key.startswith("."):
                    new_key = new_key[1:]
                condition.key = new_key
                
            # 2. If it's a nested Filter (Filter inside a Filter), recurse
            elif hasattr(condition, 'must') or hasattr(condition, 'should'):
                finalize_qdrant_keys(condition)

    # Apply to all possible Qdrant filter branches
    clean_list(filter_obj.must)
    clean_list(filter_obj.should)
    clean_list(filter_obj.must_not)
                
    return filter_obj
def get_filter_from_query(query: str): 
    metadata_info = [
        AttributeInfo(
            name="source_file",
            description="The name of the document such as 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'. Use this to filter by name of document",
            type="string"
        ),
        AttributeInfo(
            name="party_1",
            description="The name of the first legal entity or company mentioned in the contract. Use this to filter by the primary party or signer.",
            type="string"
        ),
        AttributeInfo(
            name="party_2",
            description="The name of the second legal entity or counterparty in the contract. Use this to filter for the second participant in the agreement.",
            type="string"
        ),
        AttributeInfo(
            name="contract_type",
            description="The category of the agreement, such as 'Franchise Agreement', 'NDA', 'Service Agreement', or 'Lease'. Use this when the user specifies a document type.",
            type="string"
        ),
        AttributeInfo(
            name="agreement_date",
            description="The date the contract was signed or formally created. Use for questions about when a contract was made.",
            type="string"
        ),
        AttributeInfo(
            name="effective_date",
            description="The official start date of the contract's obligations. Use for questions about when an agreement becomes active.",
            type="string"
        ),
        AttributeInfo(
            name="expiration_date",
            description="The date the contract ends or becomes invalid. Use for questions about renewals, terminations, or end dates.",
            type="string"
        ),
        AttributeInfo(
            name="governing_law",
            description="The legal jurisdiction or state laws that apply to the contract (e.g., 'California', 'New York', 'Morocco'). Use this when the user mentions a specific location or law.",
            type="string"
        ),
    ]

    document_content_description = "Detailed clauses and legal text from corporate contracts"

    groq_api_key = os.getenv("GROQ_API_KEY")
    
    # Enable JSON mode for reliable structured output
    llm = ChatGroq(
        model_name="llama-3.3-70b-versatile",
        groq_api_key=groq_api_key,
        temperature=0,
        # model_kwargs={
        #     "response_format": {"type": "json_object"}  # Force JSON output
        # }
    )
    constractor_chain = load_query_constructor_runnable(
        llm,
        document_content_description,
        metadata_info
    )
    result = constractor_chain.invoke({"query": query})
    print(f"result: {result}")
    translator = QdrantTranslator(metadata_key="")
    
    if result.filter:
        try:
            clean_filter = flatten_metadata_values(result.filter)
            print(f"clearn_filter: {clean_filter}")
            qdrant_filter = translator.visit_operation(clean_filter)
            qdrant_filter = finalize_qdrant_keys(qdrant_filter)
        except Exception as e:
            print(f"Translation failed: {e}")
            qdrant_filter = None
    else:
        qdrant_filter = None

    return qdrant_filter


In [5]:
filters = get_filter_from_query(rewriten)

result: query='Birch First Global Investments Inc. Mount Kowledge Holdings Inc.' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_1', value='Birch First Global Investments Inc.'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_2', value='Mount Kowledge Holdings Inc.')]) limit=None
clearn_filter: operator=<Operator.AND: 'and'> arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_1', value='Birch First Global Investments Inc.'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_2', value='Mount Kowledge Holdings Inc.')]


{'query': 'is this agreement done between First Global Investments Inc. and Mount Kowledge Holdings Inc in date 01/01/25 ?',
 'text': StructuredQuery(query=' ', filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Operation(operator=<Operator.OR: 'or'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_1', value='First Global Investments Inc.'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_2', value='First Global Investments Inc.')]), Operation(operator=<Operator.OR: 'or'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_1', value='Mount Kowledge Holdings Inc.'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='party_2', value='Mount Kowledge Holdings Inc.')]), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='agreement_date', value={'date': '2025-01-01', 'type': 'date'})]), limit=None)}


 ...
  Input should be a valid integer [type=int_type, input_value={'date': '2025-01-01', 'type': 'date'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type
value.str
  Input should be a valid string [type=string_type, input_value={'date': '2025-01-01', 'type': 'date'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


### hybrid search

In [ ]:

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, FusionQuery, Fusion, Filter
from fastembed import SparseTextEmbedding
from typing import Optional
from dataclasses import dataclass


# result schema
@dataclass
class SearchResult:
    chunk_id:str
    context_header:str
    metadata:dict 


class HybridSearch:
    def __init__(self, dense_model_name:str= "BAAI/bge-large-en-v1.5", sparse_model_name:str="Qdrant/bm25"):
        self.dense_model_name = dense_model_name
        self.dense_model = SentenceTransformer(self.dense_model_name, device="cuda")
        self.sparse_model_name = sparse_model_name
        self.sparse_model = SparseTextEmbedding(self.sparse_model_name)

        self.client = QdrantClient(host="localhost", port=6333)
        self.collection_name = "contracts_v1"
    def hybrid_search_with_rrf(self, query:str, filters:Optional[Filter]=None):
        """Perform hybrid search using Reciprocal Rank Fusion"""
        # rrf is a method for merging and ranking results from multiple search techniques
        query_dense_embedding = self.dense_model.encode(query,normalize_embeddings=True,batch_size=16)
        query_sparse_embeding = list(self.sparse_model.embed(query))[0]

        query_sparse_vector = models.SparseVector(
            indices=query_sparse_embeding.indices,
            values=query_sparse_embeding.values
        )
        ## hybrifd search
        results = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                Prefetch(
                    query=query_dense_embedding,
                    using="dense",
                    limit=10,
                    filter=filters
                ),
                Prefetch(
                    query=query_sparse_vector,
                    using="sparse",
                    limit=10,
                    filter=filters
                )
            ],
            query= FusionQuery(fusion=Fusion.RRF)
        )
        return self.hybrid_search_points_to_results(results.points)
    
    def hybrid_search_points_to_results(self, points):

        return [
            SearchResult(
                chunk_id=str(point.id),
                context_header = str(point.payload.get("context_header","")),
                metadata=point.payload
            )
            for point in points
        ]



In [7]:
hybrid_search = HybridSearch()
results = hybrid_search.hybrid_search_with_rrf(query=rewriten,filters=filters)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3292.63it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-02 21:05:43.728 | WARNING  | fastembed.common.model_management:download_files_from_huggingface:223 - Local file sizes do not match the metadata.


In [8]:
results

[SearchResult(chunk_id='461d72c5-631b-fdb5-cfdd-efeeb3a41af6', context_header='CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf from Type: MARKETING AFFILIATE AGREEMENT', metadata={'text': "15/16,\xa0St. Thomas,\xa0VI\xa00080\xa0(referred\xa0to\xa0as\xa0'Company')\xa0and\xa0MOUNT\xa0KNOWLEDGE\xa0HOLDINGS INC. and/or assigns, a corporation incorporated in the State of Nevada, with its main place of business located\xa0at\xa0228\xa0Park\xa0Avenue\xa0S. #56101\xa0New\xa0York,\xa0NY\xa010003\xad1502\xa0(referred\xa0to\xa0as\xa0'Marketing Affiliate'\xa0or\xa0'MA'). WHEREAS, this Agreement is to set forth in a formal agreement the prior verbal understandings between the parties in place since December 31, 2012 pertaining to the business described hereinbelow; and\n\nWHEREAS, Company, the owner of certain distribution rights to the Technology, technology and content as set forth in Exhibit A and related technical documentation (hereafter collectively referred

### rerank

In [9]:
import logging
from dataclasses import dataclass
import os 
from dotenv import load_dotenv
import cohere
from typing import Optional

load_dotenv()
logger = logging.getLogger(__name__)

@dataclass
class RerankedResult:
    chunk_id: str
    # text: str
    rerank_score: float
    metadata: dict
    original_rank: int
 

class Reranker:
    # rerank-v3.5 / rerank-v4.0
    #rerank-english-v3.0
    def __init__(self, model_name:str = "rerank-v3.5"):
        self.model_name = model_name
        cohere_api_key = os.getenv("COHERE_API_KEY")
        if not cohere_api_key:
            raise ValueError("cohere api key not found in the .env file")
        self.client = cohere.Client(cohere_api_key)
        logger.info(f"Reranker initialized - model: {model_name}")

    def rerank(self, query: str, results: list, top_n:int=8)->list:
        """
        Rerank hybrid search results using Cohere API. 
        Returns:
            list of RerankedResult sorted by rerank_score descending
        """
        if not results:
            return []
        documents = [r.metadata.get("text","") for r in results] # r.page_content for r in results

        response = self.client.rerank(
            model = self.model_name,
            query = query,
            documents = documents,
            top_n = top_n,
            return_documents = False
        )

        reranked = []
        for hit in response.results:
            original = results[hit.index]
            reranked.append(RerankedResult(
                chunk_id=original.chunk_id,
                rerank_score=hit.relevance_score,
                metadata=original.metadata,
                original_rank=hit.index + 1,
            ))
        logger.info(f"Cohere reranked {len(results)} → top {len(reranked)} results")

        return reranked




In [10]:
reranker = Reranker()
results = reranker.rerank(rewriten,results)

In [11]:
results

[RerankedResult(chunk_id='ef64973e-39b8-536e-d6a6-70c08aeb3a93', rerank_score=0.769306, metadata={'text': 'This Agreement was executed as of the date set forth above.\n\n## COMPANY MA\n\nBIRCH FIRST INVESTMENTS INC. A US Virgin Islands corporation\n\nBy: /\n\ns/ Pier S. Bjorklund\n\nPier S. Bjorklund, President\n\nMOUNT KNOWLEDGE HOLDINGS INC. A Nevada corporation\n\nBy: / s/ James D. Beatty James D. Beatty, CEO and President', 'context_header': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf from Type: MARKETING AFFILIATE AGREEMENT', 'file_type': 'pdf', 'source_file': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'contract_type': 'MARKETING AFFILIATE AGREEMENT', 'page': 14, 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\testing_2\\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'party_1': 'Birch First Global Investments Inc.', 'party_2': '

## Generation

## question answering

In [ ]:

from typing import List, Dict


SYSTEM_PROMPT = """You are a precise legal contract analyst assistant for an enterprise consulting firm.
Your job is to answer questions about contracts strictly based on the provided contract excerpts.

RULES YOU MUST FOLLOW:
1. ONLY answer using information found in the provided contract excerpts below.
2. If the answer is not in the excerpts, say exactly: "This information was not found in the provided contract documents."
3. NEVER invent dates, values, clause numbers, or party names.
5. Be concise and direct — answer in plain English, not legal jargon.
"""


def build_user_prompt(query:str, chunks:List[Dict])-> str:

    texts = [chunk.get("text","") for chunk in chunks]
    context = "\n\n".join(texts)
    return f"""{context}
    ---
    QUESTION: {query} 
    Answer based strictly on the excerpts above."""


## llm client

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import json

import os
from dotenv import load_dotenv
from torch import chunk


load_dotenv()


class LLMClient:

    def __init__(self, model_name: str = "llama-3.3-70b-versatile"):
        """Groq LLM client for contract question answering."""
        self.model_name = model_name
        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("can't find the GROQ_API_KEY in .env")
        self.llm = ChatGroq(
            model=self.model_name, groq_api_key=groq_api_key, temperature=0
        )

        print(f"[INFO] LlmClient initialized - model {self.model_name}")

    def generate_response(self, query: str, chunks):
        USER_PROMPT = build_user_prompt(query, chunks)

        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=USER_PROMPT),
        ]
        try:
            answer = self.llm.invoke(messages)
        except Exception as e:
            raise RuntimeError(f"Groq API call failed: {e}") from e

        sources = [
            {
                "file_name": chunk.get("source_file", ""),
                "file_type": chunk.get("file_type", ""),
                "page": chunk.get("page", ""),
                "contract_type": chunk.get("contract_type", ""),
                "agreement_date": chunk.get("agreement_date", ""),
                "effective_date": chunk.get("effective_date", ""),
                "expiration_date": chunk.get("expiration_date", ""),
                "agreement_date_human_display": chunk.get(
                    "agreement_date_human_display", ""
                ),
                "effective_date_human_display": chunk.get(
                    "effective_date_human_display", ""
                ),
                "expiration_date_human_display": chunk.get(
                    "expiration_date_human_display", ""
                ),
                "party_1": chunk.get("party_1", ""),
                "party_2": chunk.get("party_2", ""),
                "notice_period_to_terminate": chunk.get(
                    "notice_period_to_terminate", ""
                ),
                "renewl_term": chunk.get("renewl_term", ""),
                "governing_law": chunk.get("governing_law", ""),
                "preview": chunk["text"][0:200] + "...",
            }
            for chunk in chunks
        ]

        results = {
            "answer": answer.content,
            "sources": sources,
        }
        return self.format_result(results)
        # return results

    @staticmethod
    def format_result(results):
    # results is likely a dict containing a list under "sources"
    # or just a list itself. Adjust accordingly:
        sources_input = results.get("sources", [])
        
        sources_map = {}

        for source in sources_input:
            filename = source.get("file_name", "unknown")
            
            if filename not in sources_map:
                # First time seeing this file → create the entry
                sources_map[filename] = {
                    "filename": filename,
                    "file_type": source.get("file_type", ""),
                    "page": source.get("page", ""),
                    "contract_type": source.get("contract_type", ""),
                    "agreement_date": source.get("agreement_date", ""),
                    "effective_date": source.get("effective_date", ""),
                    "expiration_date": source.get("expiration_date", ""),
                    "agreement_date_human_display": source.get("agreement_date_human_display", ""),
                    "effective_date_human_display": source.get("effective_date_human_display", ""),
                    "expiration_date_human_display": source.get("expiration_date_human_display", ""),
                    "party_1": source.get("party_1", ""),
                    "party_2": source.get("party_2", ""),
                    "pages": [source["page"]] if source.get("page") else [],
                    "notice_period_to_terminate": source.get("notice_period_to_terminate", ""),
                    "renewl_term": source.get("renewl_term", ""),
                    "governing_law": source.get("governing_law", ""),
                    "preview": source.get("preview", ""),
                }
            else:
                # Already seen this file → update the pages list
                existing = sources_map[filename]
                page = source.get("page")
                if page and page not in existing["pages"]:
                    existing["pages"].append(page)

        return {
            "answer": results["answer"],
            "sources": list(sources_map.values()),
            }

    # ── Chunk formatter ───────────────────────────────────────────────────────
    @staticmethod
    def reranked_to_chunks(reranked_results: list) -> list[dict]:
        """
        Convert RerankedResult objects from reranker.py into
        the simple dict format expected by build_user_prompt().

        Args:
            reranked_results : list of RerankedResult from Reranker.rerank()

        Returns:
            list of dicts with keys: text, file_name, section_title, chunk_id
        """
        return [
            {
                **r.metadata,
                "chunk_id": r.chunk_id,
                "text": r.metadata.get("text", ""),
                "file_name": r.metadata.get("source", "unknown"),
                "original_score": r.original_rank,
            }
            for r in reranked_results
        ]

    # if __name__ == "__main__":


#     llm_client = LLMClient()
#     reranker = Reranker()
#     query_rewriter = QueryRewriting()
#     hybrid_search = HybridSearch()
#     original_query = "what is bla bla bla ?"

#     qdrant_filters = get_filter_from_query(original_query)
#     rewrited_query = query_rewriter.rewrite_query(original_query)
#     results = hybrid_search.hybrid_search_with_rrf(rewrited_query,fliters=qdrant_filters)
#     reranked_results = reranker.rerank(rewrited_query, results)
#     chunks = llm_client.reranked_to_chunks(reranked_results)

#     answer = llm_client.generate_response(rewrited_query,chunks)

#     print(answer)

In [27]:
llm = LLMClient()

[INFO] LlmClient initialized - model llama-3.3-70b-versatile


In [20]:
chunks = llm.reranked_to_chunks(results)

In [21]:
chunks

[{'text': 'This Agreement was executed as of the date set forth above.\n\n## COMPANY MA\n\nBIRCH FIRST INVESTMENTS INC. A US Virgin Islands corporation\n\nBy: /\n\ns/ Pier S. Bjorklund\n\nPier S. Bjorklund, President\n\nMOUNT KNOWLEDGE HOLDINGS INC. A Nevada corporation\n\nBy: / s/ James D. Beatty James D. Beatty, CEO and President',
  'context_header': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf from Type: MARKETING AFFILIATE AGREEMENT',
  'file_type': 'pdf',
  'source_file': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf',
  'contract_type': 'MARKETING AFFILIATE AGREEMENT',
  'page': 14,
  'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\testing_2\\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf',
  'party_1': 'Birch First Global Investments Inc.',
  'party_2': 'Mount Kowledge Holdings Inc.',
  'agreement_date': '2014-05-08T00:00:00Z',
  'ef

In [28]:
answer = llm.generate_response(original_query,chunks)

In [29]:
answer

{'answer': 'According to the provided contract excerpt, specifically in the section "MARKETING AFFILIATE AGREEMENT" and "Between:", the answer is: Yes, this agreement is between Birch First Global Investments Inc. and Mount Knowledge Holdings Inc. (Source: Exhibit 10.27, Marketing Affiliate Agreement, Between section).',
 'sources': [{'filename': 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf',
   'file_type': 'pdf',
   'page': 14,
   'contract_type': 'MARKETING AFFILIATE AGREEMENT',
   'agreement_date': '2014-05-08T00:00:00Z',
   'effective_date': 'Unknown',
   'expiration_date': '2014-12-31T00:00:00Z',
   'agreement_date_human_display': '05/08/14',
   'effective_date_human_display': 'Unknown',
   'expiration_date_human_display': '12/31/14',
   'party_1': 'Birch First Global Investments Inc.',
   'party_2': 'Mount Kowledge Holdings Inc.',
   'pages': [14, 1, 2, 13, 8],
   'notice_period_to_terminate': '30 days',
   'renewl_term': 'successive 1 year

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional
## enum

class ConfidenceLevel(str, Enum):
    HIGH   = "high"    
    MEDIUM = "medium" 
    LOW    = "low"

class QueryIntent(str, Enum):
    EXPIRY_DATE      = "expiry_date"       
    TERMINATION      = "termination"       
    LIABILITY        = "liability"         
    PAYMENT          = "payment"           
    GOVERNING_LAW    = "governing_law"     
    AUTO_RENEWAL     = "auto_renewal"      
    PARTIES          = "parties"           
    CONFIDENTIALITY  = "confidentiality"  
    GENERAL          = "general"           


## Cotation schema

class Citation:
    """
    Points the user to the exact source of the answer.
    Every answer MUST have at least one citation.
    """
    file_name: str = Field()
    relevant_quote: Optional[str] = Field(
        default=None,
        description="Short verbatim quote (max 100 chars) from the contract"
    )


## answer schema

 
class ContractAnswer(BaseModel):
    """
    Structured response returned by the LLM for every contract query.

    The LLM is instructed to populate all fields.
    Guardrails validates this before it reaches the API.
    """

    answer: str = Field()

    confidence: ConfidenceLevel = Field(
        description="How confident the LLM is based on the retrieved context"
    )

    citations: list[Citation] = Field(
        description="Source documents that support this answer (min 1 required)"
    )

    intent: QueryIntent = Field(
        default=QueryIntent.GENERAL,
        description="Detected intent of the user's query"
    )

    # Extracted structured data (when applicable)
    expiry_date: Optional[str] = Field(
        default=None,
        description="Extracted expiry date if query is about contract end date"
    )
    contract_value: Optional[str] = Field(
        default=None,
        description="Extracted contract value if mentioned in answer"
    )
    notice_period_days: Optional[int] = Field(
        default=None,
        description="Termination notice period in days if mentioned"
    )
    metadata:dict

    class Config:
        use_enum_values = True


In [4]:
import pandas as pd
def clean_date_human_display(raw_data:str)->str:
    # 1. Clean the string: Remove brackets and whitespace
    raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
    # 2. Convert to a datetime object
    date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
    # 3. Format it back to your desired string style (MM/DD/YY)
    if pd.notnull(date_obj):
        if date_obj.year < 1900:
            return f"{date_obj.month}/{date_obj.day}/{date_obj.year}"
        # %m/%d/%y gives you 07/19/12. 
        clean_date_str = date_obj.strftime('%m/%d/%y')
    else:
        clean_date_str = "Unknown" # Or keep it as None/nan
    return clean_date_str

## clean date for qdrant database
def clean_date_iso(raw_data:str):
    # 1. Clean the string: Remove brackets and whitespace
    raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
    # 2. Convert to a datetime object
    date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
    # 3. Format it back to ISO format (YYYY-MM-DD)
    if pd.notnull(date_obj):
        if date_obj.year < 1900:
            return f"{date_obj.month}/{date_obj.day}/{date_obj.year}"
        clean_date_str = date_obj.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        clean_date_str = "Unknown" 
    return clean_date_str

def clear_nan(raw_value):
    if pd.isna(raw_value):
        return "Unknown"
    else:
        return raw_value
    
def clear_brackets_and_nan(raw_value):
    if pd.isna(raw_value):
        return "Unknown"
    else:
        new_value =  str(raw_value).replace("[", "").replace("]", "").strip()
        if new_value == "":
            return "Unknown"
        return new_value



In [10]:
clean_date_iso(raw_data=" [2024-07-19] ")

'2024-07-19T00:00:00Z'